# report02 — 드론 3D 메쉬 — 오픈소스 CAD로 제작

**핵심.** 탐지에는 표적이 있어야 하는데 **Sionna 에는 드론 메쉬가 없다** — Sionna RT 는 씬 형상을 입력으로 받을 뿐 표적 형상을 만들어 주지 않는다. 그래서 **공개 공식 제원표의 숫자만으로** 오픈소스 CAD 라이브러리가 5종 드론(총 삼각형 155,712개)의 메쉬를 짓는다.

| 이 리포트의 척추 |  |
|---|---|
| **① Sionna 의 공백** | Sionna RT `PathSolver` 는 씬 지오메트리(삼각형 메쉬)를 **입력으로 소비**할 뿐 표적 형상을 생성하지 않는다(Sionna RT 창설논문 arXiv:2303.11103 의 설계 범위). 게다가 DJI 는 설계 CAD 를 공개하지 않아, 실물 드론을 시뮬레이션하려면 형상을 따로 마련해야 한다(§1). |
| **② 선행 연구의 방식** | 선행은 표적 형상을 두 갈래로 마련한다 — **외부 CAD·게임엔진 에셋**(LAMBDA=Cosys-AirSim UAV 모델 기하 + CADFEKO RCS[arXiv:2607.03826] · Great-X=Unreal Engine 자산[arXiv:2507.08716]) 또는 **형상을 버리고 점산란체**로 축약(Temporal-GNN[arXiv:2604.08306] · 3GPP TR38.901). 앞은 출처가 불투명하고, 뒤는 실루엣·내부 금속·프로펠러를 버린다(§2). |
| **③ 쓴 라이브러리·결합** | **재현 가능한 세 번째 길** — 공개 제원표 숫자를 표준 오픈소스 메쉬 라이브러리에 넘긴다: `shapely`(2D 단면) · `scipy`(스플라인 보간) · `trimesh`(로프트/스윕/회전체) · `manifold3d`(CSG 불리언). 자작 메쉬 엔진 없이 표준 연산만 조합해 **스펙→메쉬** 파이프라인을 만들고, 전파적으로 밝은 내부 금속(배터리·모터·PCB)까지 넣는다(§3). |
| **④ 검증** | `trimesh` 내장 검사(watertight·법선방향·퇴화면)를 **빌드 게이트**로 — 5종 전체 불량 면 0개. 외곽 상자가 공식 치수와 0% 로 맞는 것은 **구속조건 충족 확인**이고(`frame_fit_scale()` 이 바운딩박스를 공식 외곽에 강제로 맞춘다), 형상 일관성의 축은 **모터 대각**이다 — 대각은 모터 반경 입력으로 쓰이고, 외곽 피팅 배율이 그것을 흔든 뒤에도 공개 휠베이스와 ±2.3% 안에서 일관된다(§4). 실물 제품 사진과의 눈 대조도 §4, 프록시 실기체 CAD 와의 치수·σ 분포 대조는 report03(본 RCS 계산은 06~08). |

---


## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| 외곽치수·대각거리·프로펠러·무게·회전수 | DJI 공식 제품 스펙 (기체별 제원표) | 공식 제원 |
| 드론 부위별 재질 · 반사계수 \|Γ\| | 문헌값(ABS/PC 유전율) · ITU-R P.2040(metal) · 내부 재질표 | 물성 기준 |
| 삼각형 수 · 부위 수 · 외곽 오차 · 검사 결과 | outputs/report1.json (meshes 블록) | 측정값(코드 산출) |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `trimesh-cad` | CAD 모델링 (`src/cadkit.py` + `src/drone_cad.py`) — 로프트·스윕·회전체·**불리언(CSG)** | 🔴 **별도** (trimesh + manifold3d + shapely + scipy, CPU) |
| `trimesh-check` | 메쉬 검증 (`src/mesh_check.py`) — watertight · winding · 법선방향 · 퇴화면 | 🔴 **별도** (trimesh, CPU). 빌드 게이트로 회귀를 막는다 |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 얹은 **PO(물리광학 표면적분)** 가 냅니다 — Sionna 기본 solver 엔 이 산란적분이 없어 경로 이득만 줄 뿐 RCS 를 못 내기 때문입니다. 광선을 쏴 조명면·가림을 찾는 **SBR** 은 Sionna 의 **Mitsuba 3 엔진을 그대로** 쓰고, 그 위에 **PO 적분만 우리가** 얹습니다(SBR+PO).
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `trimesh` | 4.12.2 | 메쉬 CAD·**검증** — 로프트/스윕/불리언 + watertight·법선·퇴화면 검사 |
| `manifold3d` | 3.5.2 | **불리언(CSG) 엔진** — trimesh 백엔드. 겹친 파트의 내부 면을 녹여 없앤다 |
| `shapely` | 2.1.2 | 2D 단면 폴리곤(버퍼·오프셋) → 로프트 입력 |
| `scipy` | 1.18.0 | 스플라인(단면 보간·암 경로) · STFT(스펙트로그램) |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `matplotlib` | 3.11.0 | 도표 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: CPU 만 사용(메쉬 생성·검사는 GPU 불필요). 5종 전체 생성+검사 수십 초 규모.

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
# 드론 메쉬를 만들고 검사·렌더까지 다시 돌린다
~/.venvs/py312/bin/python src/build_report1.py    # → outputs/report1.json (meshes 블록)
~/.venvs/py312/bin/python src/make_notebook02.py  # → report02.ipynb
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 메쉬 생성·검사 스크립트가 `outputs/report1.json` 의 `meshes` 블록에 삼각형 수·부위 수·외곽 오차·검사 결과를 남기고, 이 생성기가 그 값을 읽어 본문에 주입합니다. 숫자가 이상하면 그 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/figures/report1_cad_pipeline.png` | 형상을 깎는 순서 도식 |
| `outputs/figures/report1_envelope.png` | 외곽 상자 치수 대조 그림 |
| `outputs/figures/report1_meshcheck.png` | 메쉬 품질 검사 결과 그림 |
| `outputs/renders/r1_30_drone_*.png` | 5종 드론 3-뷰 렌더 |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- 이 모델은 **제원표의 치수와 눈에 보이는 형상**을 맞춘 것이다. 나사·배선·틈새 같은 밀리미터 이하 디테일은 없다 — 이 리포트는 '겉모양이 맞다'까지만 보장한다.
- 실물 제품 사진과의 **눈 대조**는 §4 에서 한다. 서로 다른 프록시 실기체 CAD·커뮤니티 메쉬와의 **치수·σ 분포 대조**는 report03(본 RCS 계산은 06~08).
- **레이더 밝기(RCS, σ)** 는 이 리포트에서 계산하지 않는다. 여기서 만든 메쉬를 **재료**로 삼아 이후 리포트(06~08)가 광선+물리광학으로 밝기를 계산한다.

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| report01 — 통제 환경: 반무향 챔버 | 이 드론들을 **띄울 무대**. 앞 리포트. |
| report03 — 모델을 믿어도 되나(분포 대조) | 겉모양을 프록시 실기체 CAD·커뮤니티 메쉬와 **치수·σ 분포로 대조**한다(본 RCS 계산은 06~08). 다음 리포트. |
| report06~08 — RCS · SBR | 이 메쉬를 재료로 표적의 **레이더 밝기 σ** 를 계산한다. |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **메쉬(mesh)** | 3D 물체의 표면을 작은 **삼각형** 수만 개로 이어붙인 껍데기. 컴퓨터가 다루는 형상의 기본 형식. |
| **삼각형(면, face)** | 메쉬를 이루는 최소 조각. 많을수록 곡면이 매끈하지만 계산은 무거워진다. |
| **외곽 상자(bounding box)** | 물체를 딱 감싸는 최소 직육면체. 그 가로·세로·높이가 곧 제원표의 외곽치수. |
| **로프트(loft)** | 빵을 슬라이스로 쌓듯, 여러 **단면**을 위치별로 놓고 매끄럽게 이어 3D 덩어리를 만드는 것. |
| **스윕(sweep)** | 정해진 **경로**를 따라 단면을 밀고 나가며 만드는 것(예: 휘어진 팔·프로펠러 날). |
| **회전체(revolve)** | 옆모습 하나를 축 둘레로 **돌려** 만드는 것(예: 모터 벨, 렌즈). |
| **불리언 / CSG** | 두 덩어리를 **합치거나(union) 파내는(subtract)** 연산. 겹친 속면을 녹여 하나의 매끈한 표면으로. |
| **watertight(수밀)** | 구멍·틈 없이 **완전히 닫힌** 껍데기. 물을 부어도 안 새는 그릇. 전파 반사 계산의 전제. |
| **법선(normal)** | 각 삼각형이 **어느 쪽이 바깥인지** 가리키는 화살표. 뒤집히면 되쏘는 방향을 틀리게 계산한다. |
| **퇴화면(degenerate)** | 면적이 0에 가까운 **찌부러진 삼각형**. 계산을 망가뜨리므로 없어야 한다. |
| **\|Γ\| (반사계수)** | 그 표면이 전파를 되쏘는 정도. 0=투명, 1=완전 거울. 금속은 1에 가깝고 플라스틱은 낮다. |
| **점산란체** | 표적의 형상을 버리고 **크기 없는 점 하나**로 축약한 모델. 검출 알고리즘 연구에 흔하지만 실루엣·내부 금속·프로펠러는 사라진다. |

</details>

---


## §1. Sionna 공백 — 표적 메쉬는 입력일 뿐, Sionna 가 만들어 주지 않는다

Sionna RT 는 전파가 장면 안 물체에 부딪혀 반사·회절하는 경로를 계산하는 엔진입니다. 이때 물체의 **3D 형상(삼각형 메쉬)은 입력**입니다 — `PathSolver` 는 씬 지오메트리를 받아 전파를 튕길 뿐, 표적 드론의 형상을 스스로 만들어 주지 않습니다(Sionna RT 창설논문 arXiv:2303.11103 이 문서화한 설계 범위). 그래서 실제 드론을 시뮬레이션하려면 **그 드론의 형상을 어디선가 마련**해야 하는데, DJI 같은 제조사는 경쟁·보안 문제로 **설계 3D(CAD) 파일을 공개하지 않습니다.** 인터넷에 떠도는 무료 모델은 대개 비율이 어긋나거나, 속이 텅 비어 있거나, 출처를 알 수 없어 재현성이 없습니다.

즉 공백은 두 겹입니다 — **① Sionna 는 표적 형상을 생성하지 않고, ② 제조사도 형상을 주지 않는다.** 이 리포트가 박는 못은 그 사이를 **재현 가능하게** 메우는 것입니다: 제조사가 **반드시 공개하는** 제원표(spec sheet)의 숫자만으로 형상을 유도합니다. 제원표엔 늘 이런 값이 있습니다:

- **외곽 치수**: 펼친 기체를 감싸는 상자의 가로 × 세로 × 높이 (mm)
- **대각거리**: 마주 보는 모터 축 사이 거리 (프레임 크기의 표준 지표)
- **프로펠러**: 지름과 날 수
- **무게**·**회전수**(호버링/최대) 등

---
## §2. 선행 연구는 표적 형상을 어떻게 마련했나 — 두 갈래

이 공백은 우리만의 문제가 아니라 Sionna 를 센싱(ISAC)에 쓰는 선행이 공통으로 만나는 것입니다. 선행이 표적 형상을 마련하는 방식은 크게 둘입니다:

| 갈래 | 방식 | 대표 선행 | 한계 |
|---|---|---|---|
| **외부 CAD·에셋** | 시뮬레이터·게임엔진 자산에서 형상을 가져오고, 산란은 상용 EM 툴로 따로 구한다 | LAMBDA=**Cosys-AirSim UAV 모델 기하 + CADFEKO RCS**(arXiv:2607.03826) · Great-X=Unreal Engine 자산(arXiv:2507.08716) | 출처·라이선스가 불투명하고 스펙에서 재현되지 않는다 |
| **점산란체 근사** | 표적의 형상을 아예 버리고 **크기 없는 점 하나**로 둔다 | Temporal-GNN=점산란체(arXiv:2604.08306) · 3GPP TR38.901 | 실루엣·내부 금속·프로펠러가 사라진다 |

검출 알고리즘 연구엔 점산란체가 흔하지만, 드론의 실제 모양·속 금속·회전 프로펠러를 다뤄야 하는 우리 실험엔 형상이 필요합니다. 상용 CAD 에셋은 형상을 주지만 스펙에서 재현되지 않아 출처가 불투명합니다. 우리는 두 갈래를 모두 피하고 **공개 스펙에서 형상을 유도하는 세 번째 길**을 택합니다(§3).

---
## §3. 우리가 쓴 방식 — 오픈소스 CAD 라이브러리로 스펙에서 메쉬를 짓는다

제원표의 숫자를 표준 오픈소스 메쉬 라이브러리에 넘기면 형상이 구성됩니다. 쓰는 라이브러리는 넷입니다 — `shapely`(2D 단면 정의) · `scipy`(부드러운 곡선 보간) · `trimesh`(단면을 3D 메쉬로) · `manifold3d`(덩어리 불리언 합치기). **자작 메쉬 엔진을 만들지 않고** 어느 3D 도구에나 있는 **네 가지 표준 연산**만 조합합니다:

| 연산 | 무엇을 하나 | 드론의 어디 |
|---|---|---|
| **쌓기**(로프트) | 여러 **단면**을 높이별로 놓고 매끄럽게 잇는다 | 동체·캐노피 |
| **쓸기**(스윕) | **경로**를 따라 단면을 밀고 나간다 | 휘어진 팔·프로펠러 날 |
| **돌리기**(회전체) | 옆모습 하나를 축 둘레로 돌린다 | 모터 벨·렌즈 |
| **붙이기/파내기**(불리언·CSG) | 두 덩어리를 합치거나 파낸다 | 부품 결합·구멍 |

> **비유(직관).** 프라모델 장인이 실물 없이 **사진 몇 장 + 치수표**만으로 똑같이 깎아내는 것과 같습니다. 장인은 3D 스캐너로 본뜨지 않고 사진을 보고 비례와 특징을 손으로 재현합니다 — 우리는 그 손 대신 **코드**가, 붓 대신 **로프트·스윕·회전체**가 합니다. 그래서 스캔·다운로드 없이도 실물과 닮되 모든 치수가 공식 스펙에서 유도돼 **재현 가능**합니다.

특히 **붙이기(CSG)** 가 중요합니다. 조각들을 그냥 겹쳐 놓으면 껍데기 안에 **보이지 않는 속면**이 남아 전파 계산을 어지럽힙니다. `manifold3d` 의 불리언 합집합은 겹친 부분의 속면을 **녹여 없애고**, 바깥 표면만 남은 하나의 매끈한 다양체(manifold) 덩어리로 만들어 줍니다.

단, 이 녹이기는 **한 부위(그룹) 안**에서의 이야기입니다. **부위 사이**는 일부러 겹쳐 둡니다 — 배터리·PCB 는 동체 **안에 통째로 매몰**되고(속 금속을 넣는 설계, 아래), 동체↔캐노피 같은 이음부도 관입시켜 틈을 없앱니다. 실측으로 부위 간 겹침은 부피 합의 약 30%(기체별 0.66~37.4%, `report_mesh/outputs/mesh_verify.json` F_overlap)입니다. 겹침이 이중계산을 만들지 않는 이유는 산란적분이 **부피가 아니라 히트한 표면 기여만** 더하기 때문입니다 (불투명 금속면에선 광선이 멈추고, 반투명 플라스틱 셸은 투과해 내부 금속까지 봅니다 — →report07) 이 내부 겹침은 계산을 어지럽히지 않습니다 — 같은 이유로 부위 부피의 단순 합산은 이중계산이 됩니다.

부위별로 이름표(그룹)를 붙여 조립합니다 — **동체(body) · 캐노피(canopy) · 팔(arm) · 모터(motor) · 프로펠러(prop) · 착륙장치(gear) · 카메라(camera) · 배터리(battery) · 기판(pcb) · 식별색(accent)**. 이 이름표는 뒤에서 부위마다 다른 **재질**(전파를 얼마나 되쏘는지)을 붙일 때 그대로 쓰입니다.

![CAD 파이프라인](outputs/figures/report1_cad_pipeline.png)

> 위 그림은 단면 하나가 어떻게 쌓기·다듬기를 거쳐 완성된 동체 표면이 되는지를 단계별로 보여줍니다.

### 그 '실제와 닮은 모양'은 어디서 왔나 — 세 층의 사실성

완성된 모델은 실물과 꽤 닮았지만, 이 닮음은 외부 3D 모델을 가져와 만든 것이 아닙니다 — 어떤 다운로드 에셋도 본떠 오지 않았습니다(§2). 닮음은 **세 층이 쌓여** 만들어집니다:

| 층 | 무엇이 | 어떻게 '닮게' 만드나 | 어디서 왔나 |
|---|---|---|---|
| **① 정량층** (숫자) | 공식 치수 — 대각거리·외곽 L×W×H·프로펠러 지름 | **크기와 비율**을 강제한다. 비율이 맞으면 실루엣의 절반은 이미 맞다 | DJI 공식 스펙 → `DroneSpec`(§1) |
| **② 정성층** (형태) | 제품 **사진·3면도를 사람이 보고 뽑은 형태 특징** | "마빅=눈물방울 동체+등 배터리+렌즈 3개 짐벌", "팬텀=고정 팔+착륙다리", "미니=앞뒤 로터 높이차" 같은 관찰을 **코드 파라미터로 번역** | 사람이 사진을 보고 값을 정함 → 아래 |
| **③ 검증층** (사후) | 실기체 스캔·타사 실물 CAD 와 **나중에** 대조 | 만든 뒤 닮았는지 **측정으로 확인** | report03·mesh08 |

**핵심은 ②입니다.** '모양'은 어딘가에서 복사한 게 아니라, **사람이 실물 사진을 보고 '이 기종은 이렇게 생겼다'를 몇 개의 숫자·선택지로 적어 넣은 것**입니다. 그 선택지가 코드에 이렇게 들어 있습니다:

```python
# src/drones.py 의 DroneSpec — 기종별 '형태 파라미터' (공식 스펙 아님, 사진 관찰값)
mavic4pro:  gimbal_style='triple'   gear='none'   rotor_deg=(32,148,212,328)  body_lw=(1.52,0.62)
phantom4 :  gimbal_style='recessed' gear='legs'   fixed_arm=True              body_lw=(1.15,0.85)
mini5pro :  gimbal_style='single'   gear='none'   rotor_z_mm=(-12,+2,+2,-12)  # 앞뒤 로터 높이차
```

그리고 `src/drone_cad.py` 안에는 그 파라미터를 **실제 곡면으로 바꾸는 함수들**이 있습니다 — "마빅 동체는 눈물방울"이라는 관찰을 `_body_folding`(초타원 단면을 로프트)이, "팔이 접힌다"를 `_arm_folding`(스윕)이, "짐벌이 3렌즈"를 `_gimbal_infinity`가, "착륙다리"를 `_gear_skids`/`_gear_tall` 이 만듭니다. 즉 **모양의 사실성 = (스펙 비율) × (사진에서 뽑은 형태 파라미터) × (곡면 프리미티브)**입니다.

> ⚠️ **그래서 무엇을 보장하고 무엇은 아닌가.** 보장: **비례·크기·주요 형태 특징**이 실물과 닮았고, 공식 외곽에 정확히 맞춰져 있으며(구속조건 — §4), 흔들린 뒤에도 남는 모터 대각 일관성과 사후 스캔 대조를 통과(report03). 비보장: 나사·배선·미세 곡률 같은 **밀리미터 이하 디테일** — 사진에서 안 보이는 것은 넣지 않았습니다. 레이더 밝기(RCS)에는 이 정도 형태면 충분합니다(파장 86 mm 앞에서 mm 디테일은 안 보인다, report06).

### 재현한 5종

이렇게 해서 크기와 용도가 서로 다른 **5종**을 재현했습니다. 손바닥만 한 250 g 미니 드론부터, 8개 로터에 9.5 kg 나 나가는 산업용 대형 옥토콥터까지 폭이 넓습니다. 표적의 **크기와 프로펠러 수**에 따라 레이더에 잡히는 양상이 달라지므로, 일부러 다양하게 갖췄습니다.

| 드론 | 로터 | 프로펠러 | 무게 | 삼각형 수 | 부위 수 |
|---|---|---|---|---|---|
| DJI Mini 5 Pro | 4 | 152 mm · 2날 | 250 g | 27,030 | 22 |
| DJI Mavic 4 Pro | 4 | 267 mm · 2날 | 1063 g | 28,548 | 18 |
| DJI Matrice 4E | 4 | 274 mm · 2날 | 1219 g | 31,712 | 25 |
| DJI Phantom 4 | 4 | 240 mm · 2날 | 1380 g | 28,160 | 24 |
| DJI S1000+ | 8 | 381 mm · 2날 | 9500 g | 40,262 | 40 |

5종을 합치면 삼각형이 총 **155,712개**입니다. 삼각형이 많을수록 곡면이 매끈해 전파 되쏘기를 정밀하게 계산할 수 있지만, 그만큼 계산이 무거워집니다 — 이 정도가 형상 충실도와 계산 부담의 균형점입니다.

아래는 각 드론을 세 방향(비스듬히 · 옆 · 위)에서 렌더한 모습입니다.


**DJI Mini 5 Pro**

| 정면 | 비스듬히 | 옆 | 위 |
|---|---|---|---|
| ![mini5pro front](outputs/renders/r1_30_drone_mini5pro_front.png) | ![mini5pro iso](outputs/renders/r1_30_drone_mini5pro_iso.png) | ![mini5pro side](outputs/renders/r1_30_drone_mini5pro_side.png) | ![mini5pro top](outputs/renders/r1_30_drone_mini5pro_top.png) |

![.](outputs/renders/anim/spin_mini5pro.gif)

<sub>Mini 5 Pro 3D 모델 회전.</sub>

**DJI Mavic 4 Pro**

| 정면 | 비스듬히 | 옆 | 위 |
|---|---|---|---|
| ![mavic4pro front](outputs/renders/r1_30_drone_mavic4pro_front.png) | ![mavic4pro iso](outputs/renders/r1_30_drone_mavic4pro_iso.png) | ![mavic4pro side](outputs/renders/r1_30_drone_mavic4pro_side.png) | ![mavic4pro top](outputs/renders/r1_30_drone_mavic4pro_top.png) |

![.](outputs/renders/anim/spin_mavic4pro.gif)

<sub>Mavic 4 Pro 3D 모델 회전(실측 실험용 드론).</sub>

**DJI Matrice 4E**

| 정면 | 비스듬히 | 옆 | 위 |
|---|---|---|---|
| ![matrice4e front](outputs/renders/r1_30_drone_matrice4e_front.png) | ![matrice4e iso](outputs/renders/r1_30_drone_matrice4e_iso.png) | ![matrice4e side](outputs/renders/r1_30_drone_matrice4e_side.png) | ![matrice4e top](outputs/renders/r1_30_drone_matrice4e_top.png) |

![.](outputs/renders/anim/spin_matrice4e.gif)

<sub>Matrice 4E 3D 모델 회전(실측 실험용 드론).</sub>

**DJI Phantom 4**

| 정면 | 비스듬히 | 옆 | 위 |
|---|---|---|---|
| ![phantom4 front](outputs/renders/r1_30_drone_phantom4_front.png) | ![phantom4 iso](outputs/renders/r1_30_drone_phantom4_iso.png) | ![phantom4 side](outputs/renders/r1_30_drone_phantom4_side.png) | ![phantom4 top](outputs/renders/r1_30_drone_phantom4_top.png) |

![.](outputs/renders/anim/spin_phantom4.gif)

<sub>Phantom 4 3D 모델 회전.</sub>

**DJI S1000+**

| 정면 | 비스듬히 | 옆 | 위 |
|---|---|---|---|
| ![s1000plus front](outputs/renders/r1_30_drone_s1000plus_front.png) | ![s1000plus iso](outputs/renders/r1_30_drone_s1000plus_iso.png) | ![s1000plus side](outputs/renders/r1_30_drone_s1000plus_side.png) | ![s1000plus top](outputs/renders/r1_30_drone_s1000plus_top.png) |

![.](outputs/renders/anim/spin_s1000plus.gif)

<sub>S1000+ 3D 모델 회전(큰 옥토콥터).</sub>

### 색은 곧 재질 — 그리고 실제 크기 비교

위 렌더의 **색은 재질**입니다(모든 드론 공통 규약): **파랑=금속**(모터·배터리 포일) · **회색=플라스틱**(셸·착륙장치·프로펠러 — 같은 플라스틱, 프로펠러는 얇아 &#124;Γ&#124; 가 조금 낮다) · **검정=탄소섬유**(암) · **주황=카메라**(금속하우징+유리 복합체) · **초록=PCB**. 색만 보면 그 부위가 무슨 재질인지 알 수 있고, 이 재질이 그대로 RCS(되쏘는 밝기) 계산에 쓰입니다(→ report06·07).

### 프로펠러는 평판이 아닙니다 — 진짜 익형, 그리고 모델마다 다릅니다

프로펠러는 레이더에 특히 중요합니다(빠르게 돌아 **마이크로도플러**를 만드는 유일한 부위 → report08). 그래서 납작한 판이 아니라 **실제 익형**으로 구성했습니다 — NACA-4 익형 단면을 스팬을 따라 로프트합니다:

- **단면 = NACA-4 계열 익형**(앞전 둥글고 뒷전 뾰족한 진짜 날개 단면)을 스팬을 따라 **로프트**
- **테이퍼**: 루트는 좁고 30% 지점이 가장 넓다가(최대 시위 ≈ 0.26·R) 팁에서 다시 좁아짐
- **워시아웃 트위스트**: 피치각이 루트에서 크고 팁으로 갈수록 작아짐
- **시미터 스윕**: 회전 방향으로 살짝 휨

그리고 **모델마다 다릅니다** — 스펙에서 **반경·날개 수·피치**를 받습니다. 피치는 1회전당 전진량이며, 각 반경에서의 날개 각도는 **θ(r) = arctan(P / 2πr)** 로 정해집니다(루트는 가파르고 팁은 완만):

| 드론 | 프로펠러 지름 | 피치 | 최대 시위 | 팁 각도 θ(R) |
|---|---|---|---|---|
| Mini 5 Pro | 152 mm | 2.8" | 19.8 mm | 8.4° |
| Phantom 4 | 240 mm | 5.0" | 31.2 mm | 9.6° |
| Mavic 4 Pro | 267 mm | 5.8" | 34.7 mm | 10.0° |
| Matrice 4E | 274 mm | 5.7" | 35.6 mm | 9.5° |
| S1000+ | 381 mm | 5.2" | 49.5 mm | 6.3° |

<sub>피치가 다르면 같은 회전수에서도 날개가 전파를 되쏘는 각도가 달라집니다 — 마이크로도플러 서명이 모델마다 다른 이유 중 하나입니다.</sub>

**실제 크기 비교(같은 축척, 위에서 본 모습).** 5종은 크기가 크게 다릅니다 — Mini 5 Pro(275 mm)부터 S1000+(1045 mm)까지 대각 길이가 약 4배:

![drone size comparison](outputs/figures/drone_size_compare.png)

<sub>같은 축척으로 나란히 둔 5종 실루엣(재질색·스케일바 0.5 m). 크기가 다르면 되쏘는 밝기(RCS)도 달라진다 — 큰 S1000+ 가 작은 Mini 5 Pro 보다 훨씬 밝게 잡힌다.</sub>

**부위별로 따로 움직인다 — 분절(articulated) 메쉬.** 몸체와 프로펠러가 **독립적으로** 회전합니다. 아래는 5종이 몸체를 돌리며 동시에 프로펠러를 스핀시키는 모습입니다(같은 메쉬로 마이크로도플러 시뮬레이션을 할 수 있는 이유 → report08):

![five drones spinning](outputs/renders/anim/drone_gallery_row.gif)

<sub>5종 동시 회전 + 프로펠러 스핀(분절 메쉬). 몸체 자세와 블레이드 회전이 분리돼 있어, 실제 비행 중 프로펠러만 빠르게 도는 상황을 그대로 만들 수 있다.</sub>

### 껍데기보다 속 — 왜 내부 금속을 넣나

**눈에 보이는 것**과 **전파에 보이는 것**은 다릅니다. 우리 눈에 드론은 매끈한 플라스틱 몸통이지만, 레이더(전파)의 눈으로 보면 그 플라스틱 껍데기는 상당히 **반투명**합니다 — 전파의 일부가 껍데기를 통과합니다. 정작 강하게 되빛나는 건 **속에 든 금속**입니다. 그래서 껍데기만 만들지 않고, 전파적으로 밝은 **내부 금속 부품**을 함께 넣습니다:

- **배터리 팩** — GHz 대역에서 파우치의 금속 포일은 사실상 금속판처럼 되쏩니다.
- **모터** — 구리 코일을 감은 금속 벨.
- **회로기판(PCB)** — FR-4 밑판에 넓은 **구리 접지면**이 깔려 있어 금속면처럼 반사합니다.

부위마다 얼마나 되쏘는지를 **반사계수 |Γ|**(0=투명, 1=완전 거울)로 나타내면, 겉과 속의 차이가 한눈에 보입니다. |Γ| 는 재질의 전파 물성 **(유전율 ε_r, 도전율 σ)** 에서 **특정 주파수**로 유도된 값이라, 뒤에서 재현할 수 있도록 그 두 물성과 **평가 주파수 f = 3.5 GHz** 를 함께 적습니다:

| 부위 | 재질 | ε_r | σ [S/m] | 평가 f | &#124;Γ&#124; |
|---|---|---|---|---|---|
| 동체·캐노피·착륙장치 | 플라스틱(ABS/PC) | 2.7 | 0.02 | 3.5 GHz | 0.28 |
| 프로펠러 | 얇은 플라스틱(ABS/PC) | 2.7 | 0.02 | 3.5 GHz | 0.25 |
| 팔(arm) | 탄소섬유 | 5.0 | 3.0×10³ | 3.5 GHz | 0.90 |
| 모터·배터리 | 금속(ITU PEC) | 1.0 | 1.0×10⁷ | 3.5 GHz | 0.9998 |
| 기판(PCB) | 구리 접지면(ITU) | 1.0 | 1.0×10⁷ | 3.5 GHz | 0.80 |

<sub>물성 출처: **금속·PCB** = ITU-R P.2040 'metal'(PEC 근사, ε_r=1·σ=10⁷ S/m; 주파수 자동보정) · **플라스틱/프로펠러** = 문헌 ABS/PC(ε_r≈2.7) · **탄소섬유** = 도전성 복합(ε_r=5.0·σ=3×10³). 표의 |Γ| 는 이 (ε_r, σ)+f 에서 나온 **PO 실효값**이다 — 금속 0.9998 은 벌크 프레넬 값이고, 플라스틱 0.28·프로펠러 0.25·PCB 0.80 은 박막 간섭·복합 조립(구리면+비도체 개구)을 반영한 대표 실효값이다(단일 진리원 `src/materials.py`, Sionna RT 도 같은 표를 읽는다). f 를 명기하는 이유: |Γ| 는 주파수에 따라 달라지므로 f 없이는 표 자체로 재현이 안 된다.</sub>

숫자로 보면 내부 금속(≈1.0)이 플라스틱 껍데기(0.28)보다 **서너 배** 밝게 되쏩니다. 겉껍데기만 넣고 계산하면 표적이 실제보다 어둡게 나와, 나중에 '이 드론을 레이더로 잡을 수 있나'라는 질문에 **틀린 답**을 내게 됩니다. 그래서 속을 넣는 것이 선택이 아니라 필수입니다.

참고로 이 내부 금속 부품이 차지하는 삼각형은 DJI Mini 5 Pro 약 2,052개, DJI S1000+ 약 4,056개 수준으로, 겉모양뿐 아니라 속까지 형상을 갖췄음을 보여줍니다.

> 참고: 여기서 |Γ| 를 소개하는 이유는 **왜 속을 넣는지**를 설명하기 위해서입니다. 이 재질값으로 실제 표적 밝기(RCS, σ)를 계산하는 일은 report06~08 이 맡습니다.

---
## §4. 검증 — 실제 사진 대조 · 겉치수 · 속 · 수밀

만든 형상이 믿을 만한지 확인합니다 — **실제 제품 사진과 눈으로 대조**, **외곽 크기·비율이 공식 치수라는 구속조건을 실제로 만족하는지**, **외곽 피팅이 흔든 뒤에도 남는 모터 대각의 일관성**, **메쉬 품질**(수밀·법선·퇴화면), 그리고 **곡면 이산화가 선행 기준만큼 촘촘한지**(E²/(Rλ)). 프록시 실기체 CAD 대조는 report03 소관입니다.

### 실제 제품 사진과 대조

각 드론의 **실제 제품 사진**(왼쪽)과 우리가 **스펙시트에서 만든 메쉬를 같은 정면 각도에서 렌더한 모습**(오른쪽)을 나란히 놓았습니다. 스펙 치수(대각·프롭·엔벨로프)는 공식값 그대로 두고, 실사진·웹조사를 참고해 **결정적 형상**을 담았습니다 — Mavic 4 Pro 의 큰 전면 Hasselblad 3렌즈 짐벌·전방향 어안·전방 LiDAR, Matrice 4E 의 전면 측량 짐벌·상단 RTK 돔·레이저 거리계, Mini 5 Pro 의 전면 1인치 짐벌·전방 LiDAR·전방향 어안, Phantom 4 의 아치형 착륙다리·벨리 짐벌·5방향 비전, S1000+ 의 8로터 방사형 접이암·상단 GPS.

![real photo vs our mesh](outputs/figures/report2_photo_compare.png)

<sub>색은 **재질 규약**(plastic=회색·metal=파랑·camera=주황…)이라 실제 도색(그레이/화이트)과 다르다 — 우리가 맞추는 것은 **형상**이지 색이 아니다. 재질도 실제 구성을 반영했다(웹조사 확인): 동체 셸·프로펠러는 플라스틱(저반사), 암은 탄소섬유(준금속·강반사), 모터·배터리·PCB·짐벌 마운트는 금속(강반사) — 이 금속 내부가 RCS 를 지배한다(report06·08).</sub>

### 겉모양이 맞나 — 외곽 상자(구속조건 충족 확인)

가장 먼저 **크기와 비율**이 실물과 같은지 봅니다. `trimesh` 로 모델을 **딱 감싸는 최소의 상자**(외곽 상자)를 재서, 그 가로·세로·높이를 **공식 제원표의 외곽치수와 비교**합니다.

| 드론 | 공식 외곽 (가로×세로×높이, mm) | 만든 모델 외곽 (mm) | 최대 오차 |
|---|---|---|---|
| DJI Mini 5 Pro | 미공개 × 미공개 × 91 | 275 × 378 × 91 | 0% |
| DJI Mavic 4 Pro | 329 × 390 × 135 | 329 × 390 × 135 | 0% |
| DJI Matrice 4E | 307 × 388 × 150 | 307 × 388 × 150 | 1.9e-14% |
| DJI Phantom 4 | 290 × 290 × 196 | 290 × 290 × 196 | 0% |
| DJI S1000+ | 1016 × 1016 × 380 | 1016 × 1016 × 380 | 0% |

> ⚠️ **이 0% 는 검증 결과가 아니라 구속조건입니다.** `frame_fit_scale()`(`src/drones.py:274`)이 프레임 바운딩박스가 공식 외곽과 같아지도록 **축별 배율을 강제로 겁니다.** 따라서 위 표는 '용케 맞았다'가 아니라 **설계 의도(공식 치수)가 결과물에 실제로 새겨졌는지 확인하는 빌드 게이트**로 읽어야 하며, 형상 충실도의 독립 증거로 인용해서는 안 됩니다. 공식치수가 없는 축(Mini 5 Pro 의 가로·세로)은 애초에 구속되지 않아 표에서 '미공개'로 남습니다.

**그럼 무엇이 흔들리는 축인가 — 모터 대각거리.** 대각은 우리가 넣는 값입니다: `src/drones.py:228` 이 `diag = spec.diagonal_mm/1000` 을 읽어 `r = diag/2` 를 **모터 반경 입력**으로 씁니다. 그런데 그 뒤 외곽 피팅이 **축마다 다른 배율**(비등방)을 모터 좌표에 함께 걸기 때문에, 완성된 메쉬에서 마주 보는 모터 사이 거리는 **입력값과 같지 않습니다.** 그래도 공개 휠베이스와 **±2.3%** 안에서 일관됩니다:

| 드론 | 공개 대각 (mm) | 모델 대각 (mm) | 오차 |
|---|---|---|---|
| DJI Matrice 4E | 438.8 | 428.7 | -2.30% |
| DJI S1000+ | 1045.0 | 1043.5 | -0.14% |
| DJI Phantom 4 | 350.0 | 356.9 | +1.98% |

즉 이 일치는 '입력하지 않은 값이 맞았다'가 아니라 **'입력을 흔든 뒤에도 남는 일관성'**입니다 — 외곽을 강제로 맞추느라 프레임을 비등방으로 늘렸는데도 내부 비율이 공개 휠베이스에서 몇 % 이상 벗어나지 않았다는 뜻입니다.

<sub>⚠ 다만 <b>세 행이 같은 강도의 증거는 아닙니다</b>. 면내(X·Y) 피팅 배율이 <b>비등방</b>이라 대각이 실제로 교란되는 것은 DJI Matrice 4E(×0.851, ×1.089) 뿐이고, DJI S1000+·DJI Phantom 4 는 면내 배율이 등방이라 모델 대각 = 입력 대각 × 그 배율이 되어 <b>오차가 곧 배율 편차</b>입니다. 그리고 실제로 가장 크게 어긋나는 행이 비등방 쪽입니다(-2.30%) — 위의 ±2.3% 는 그 값입니다.</sub>

<sub>이 표에서 <b>DJI Mavic 4 Pro·DJI Mini 5 Pro 는 뺐습니다</b> — 자기참조라 증거가 되지 않는다. Mavic 4 Pro 는 DJI 공개 대각(400 mm)이 공식 외곽 328.7×390.5 mm 와 기하학적으로 모순이라 여기 쓰는 441 mm 자체가 <b>우리가 외곽에서 역산한 값</b>이고(`src/drones.py:120-129`), Mini 5 Pro 는 DJI 가 대각을 공개하지 않는 데다 면내 피팅 배율이 (1.0, 1.0) 이라 모델 대각이 입력값과 같아지는 것이 <b>대수 항등식</b>이다(오차 +0.00%). 참고로 둘을 넣어도 최대 오차는 2.30% 로 달라지지 않는다.</sub>

**Mini 5 Pro 에는 피팅이 건드리지 않은 축이 둘 있습니다.** DJI 가 이 기종에 공개한 언폴드 치수는 **프롭까지 포함한** 304 × 380 × 91 mm 인데, 우리가 강제한 건 **높이 91 mm 하나뿐**입니다(면내 배율 1.0). 가로·세로는 로터 좌표가 정하게 두었는데, 완성된 모델의 **프로펠러 회전 디스크 외곽**을 재보면 305 × 381 mm — 공식값과 **0.32% 이내**입니다. 위 표의 0% 가 구속조건 충족 확인인 것과 달리, 이 두 축은 외곽 피팅이 손대지 않은 자유 축입니다.

<sub>단 이것을 완전한 독립 증거라고 부르지는 않는다 — 로터 좌표(±76, ±114 mm)는 조사로 정한 값이고, 거기에 프롭 반경 76.2 mm 를 더하면 공식 언폴드 외곽이 거의 그대로 재구성된다((76+76.2)×2 = 304.4 mm, (114+76.2)×2 = 380.4 mm). 그 좌표의 출처가 같은 제원표라면 이 일치는 독립 검증이 아니라 재산출이다.</sub>

<sub>프롭 포함 높이라는 점이 중요하다 — 프레임만 91 mm 에 맞추면 프로펠러가 그 위로 더 얹혀 전체 높이가 105 mm 가 된다(현재 메쉬의 프롭 적층분 = 전체 − 프레임). 그래서 이 기종은 프레임이 아니라 **완성된 드론 전체**를 공식값과 견준다(현재 전체 높이 91.0 mm). 정적 메쉬의 가로·세로 bbox 는 블레이드가 어느 방위에 멈춰 있느냐에 따라 달라지므로(2날 프롭은 원반이 아니라 선이다), 공식 프롭포함 L/W 와 견줄 값은 **회전 디스크 외곽**이다.</sub>

![외곽 대조](outputs/figures/report1_envelope.png)

### 새지 않는 그릇인가 — trimesh 품질 검사

**전파 되쏘기 계산은 메쉬의 표면 상태에 민감**합니다. 특히 세 가지가 어긋나면 밝기 계산이 조용히 틀어집니다:

- **수밀(watertight)** — 껍데기에 **구멍이나 틈**이 있으면 안 됩니다. 새는 곳이 있으면 그 자리에서 전파가 표면 안팎을 넘나들며 계산이 엉킵니다.
- **법선(normal) 방향** — 각 삼각형에는 '어느 쪽이 바깥'인지 가리키는 화살표가 있습니다. 이게 **안쪽을 향해 뒤집혀** 있으면, 되쏘는 방향을 반대로 계산해 표적이 어둡거나 밝게 잘못 나옵니다.
- **퇴화면(degenerate)** — 면적이 0에 가까운 **찌부러진 삼각형**은 계산에서 0으로 나누는 오류를 냅니다.

그래서 메쉬를 만들 때마다 `trimesh` 의 **내장 검사**로 걸러냅니다 — `is_watertight`(닫힘)·`is_winding_consistent`(면 방향 일관)·법선 방향·퇴화면 판정은 모두 라이브러리가 제공하는 표준 점검입니다. 부위별로 닫혀 있는지, 안쪽을 향한 법선이 몇 개인지, 와인딩이 뒤집힌 면·찌부러진 면이 몇 개인지를 세어 **하나라도 있으면 빌드를 통과시키지 않습니다.**

결과는 깔끔합니다 — **5종 전체에서 안쪽 법선·역와인딩·퇴화면이 모두 0개**입니다. 모든 부위가 닫힌 껍데기이고, 모든 삼각형이 바깥을 봅니다. 즉 이후 리포트가 이 메쉬 위에서 하는 전파·밝기 계산이 형상 결함 때문에 틀어질 걱정은 없습니다.

![메쉬 검사](outputs/figures/report1_meshcheck.png)

### 삼각형이 충분히 촘촘한가 — 곡면 이산화 품질 E²/(Rλ)

메쉬가 닫혀 있어도, 곡면을 **너무 성기게** 쪼개면 되쏘기 계산이 틀어집니다. 얼마나 촘촘해야 충분한가에는 선행 기준이 있습니다 — 곡면을 패싯으로 쪼갤 때의 품질 지표 **E²/(Rλ)**(E=패싯 모서리 길이, R=국소 곡률반경, λ=파장)입니다.

> Ziganshin 외, *Ray-Based Simulation of Scattering from Discretized Curved Bodies* (arXiv:2604.05991) 은 이 값을 *"a physically motivated measure of discretization quality"* 로 제안합니다 — §V-C 에서 후방산란 정확도가 **E²/(Rλ) ≈ 0.5 부근에서 수렴(개선이 미미해지는 무릎)** 하고, 자신들의 차량(i-MiEV) 메쉬는 **0.4–0.6**(3 GHz) 라고 보고합니다(별개로 UTD 유효조건 은 E > 1.5λ). ⚠ 원문은 최적 구간이 **응용·산란기구에 따라 달라 경험적으로 정해야 한다**고 명시하므로, ≈0.5 는 보편 임계값이 아니라 그들의 후방산란 수렴 무릎으로 읽습니다.

**우리 5기종 드론 메쉬의 per-mesh E²/(Rλ) 값은 아직 산출하지 않았습니다.** 이 지표는 facet 별 국소 곡률반경 R 을 요구하는데(원문 §III-C: s∼E²/R), 원곡면을 모르는 임의 메쉬에서 R 추정은 열린 문제라 다음 단계(P1) 계산으로 남깁니다 — report07 §6 도 같은 이유로 보류합니다. 지금 정량적으로 말할 수 있는 것은 **R 이 정확히 알려진 정준 구 앵커** 하나입니다:

| 대상 | 지표 | 값 | vs 무릎 0.5 |
|---|---|---|---|
| 정준 구 (r=0.5 m, uv_sphere 180 seg) | E²/(Rλ) @3.5 GHz | **0.0071** | 약 70배 촘촘 |

정준 구는 R=0.5 m 가 정확히 알려져 근사식·교정상수 없이 닫힌형으로 계산됩니다(E = 2πr/180 = 17.45 mm, λ@3.5 GHz = 85.65 mm → 0.0071). 이 값이 후방산란 수렴 무릎보다 약 70배 아래이므로, **우리 되쏘기 파이프라인이 검증에 쓰는 구 프리미티브의 이산화는 병목이 아닙니다.** 드론 메쉬 삼각형은 이보다 성길 수 있으나, 뒤 리포트가 실제로 관측하는 RCS 불확실도의 지배 요인은 이산화가 아니라 **재질·자세·바이스태틱 기하**이고 광선격자 수렴(부록 report_mesh 참조)이 진짜 놉입니다.

<sub>문헌 상수 ≈0.5·0.4–0.6·1.5λ 는 Ziganshin arXiv:2604.05991 원문 PDF(§V-B/§V-C)로 직접 확인했습니다. 정준 구 0.0071 은 PRIOR_WORK_COMPARISON.md:441 에 이미 산출·문서화된 값이며, 여기 표의 숫자는 손으로 적은 것이 아니라 그 문서 상수를 f-string 으로 주입한 것입니다. 5기종 per-mesh 원장은 국소 곡률 R 산출 패스가 나오면 채웁니다(P1).</sub>

---

> **앞 리포트**: report01 — 통제 환경(반무향 챔버). 이 드론들을 **띄울 무대**를 지었습니다.
> **다음 리포트**: report03 — 여기서 만든 겉모양을 **실물과 대조**해, 이 모델을 믿어도 되는지 확인합니다.